In [2]:
import os
import urllib.request
import zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# Create Medallion Data Lake folders
for folder in ["bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)

# Download and extract the Book-Crossing dataset
DATASET_URL = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
ZIP_FILE = "book-crossings.zip"
RAW_DIR = "raw_data"

if not os.path.exists(ZIP_FILE):
    # Add a user-agent to prevent 403 Forbidden errors
    req = urllib.request.Request(
        DATASET_URL,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(ZIP_FILE, 'wb') as out_file:
        out_file.write(response.read())

with zipfile.ZipFile(ZIP_FILE, "r") as zf:
    zf.extractall(RAW_DIR)

# Load raw CSVs with semicolon separator and latin-1 encoding; save untouched to bronze/
ingestion_time = datetime.now(timezone.utc).isoformat()

raw_files = {
    "users": "BX-Users.csv",
    "books": "BX-Books.csv",
    "ratings": "BX-Book-Ratings.csv",
}

for table_name, csv_name in raw_files.items():
    csv_path = os.path.join(RAW_DIR, csv_name)
    df_bronze = pd.read_csv(
        csv_path,
        sep=";",
        encoding="latin-1",
        on_bad_lines="skip",
        low_memory=False
    )
    # Preserve raw strings cleanly for Parquet serialization without altering values
    for col in df_bronze.select_dtypes(include=["object"]).columns:
        df_bronze[col] = df_bronze[col].astype(str).replace("nan", np.nan)

    df_bronze["ingestion_timestamp"] = ingestion_time
    out_path = os.path.join("bronze", f"{table_name}.parquet")
    df_bronze.to_parquet(out_path, index=False)
    print(f"Saved {out_path} -> Shape: {df_bronze.shape}")

Saved bronze/users.parquet -> Shape: (278858, 4)
Saved bronze/books.parquet -> Shape: (271360, 9)
Saved bronze/ratings.parquet -> Shape: (1149780, 4)


In [5]:
bronze_users = pd.read_parquet("bronze/users.parquet")
bronze_books = pd.read_parquet("bronze/books.parquet")
bronze_ratings = pd.read_parquet("bronze/ratings.parquet")

# 1. Shape, data types, and missing-value counts
for label, df in [("Users", bronze_users), ("Books", bronze_books), ("Ratings", bronze_ratings)]:
    print(f"\n=== {label} Table | Shape: {df.shape} ===")
    info_df = pd.DataFrame({
        "Data_Type": df.dtypes,
        "Missing_Count": df.isna().sum(),
        "Missing_Pct": (df.isna().mean() * 100).round(2)
    })
    print(info_df)

# 2. Distribution of ratings (noting 0s)
print("\n=== Rating Score Distribution ===")
zero_count = (bronze_ratings["Book-Rating"] == 0).sum()
total_ratings = len(bronze_ratings)
print(f"Rating == 0 (Implicit): {zero_count:,} / {total_ratings:,} ({zero_count / total_ratings:.2%})")
print(f"Rating 1-10 (Explicit): {total_ratings - zero_count:,} / {total_ratings:,} ({(total_ratings - zero_count) / total_ratings:.2%})")
print(bronze_ratings["Book-Rating"].value_counts().sort_index())

# 3. Distribution of user ages and publication years
print("\n=== Age & Publication Year Anomalies ===")
age_series = pd.to_numeric(bronze_users["Age"], errors="coerce")
bad_ages = age_series[(age_series < 5) | (age_series > 100)]
print(f"Users with Age < 5 or > 100: {len(bad_ages):,} (Min: {age_series.min()}, Max: {age_series.max()}, Median: {age_series.median()})")

year_series = pd.to_numeric(bronze_books["Year-Of-Publication"], errors="coerce")
bad_years = bronze_books[(year_series <= 0) | (year_series > 2026) | (year_series.isna())]
print(f"Books with Year <= 0, > 2026, or non-numeric: {len(bad_years):,} (Min: {year_series.min()}, Max: {year_series.max()})")

# 4. Number of ratings per user and per book (explicit ratings)
explicit_ratings = bronze_ratings[bronze_ratings["Book-Rating"] > 0]
ratings_per_user = explicit_ratings.groupby("User-ID").size()
ratings_per_book = explicit_ratings.groupby("ISBN").size()

print("\n=== Explicit Ratings per User & per Book ===")
print(f"Per User -> Mean: {ratings_per_user.mean():.2f}, Median: {ratings_per_user.median()}, 90th Percentile: {np.percentile(ratings_per_user, 90):.1f}, Max: {ratings_per_user.max()}")
print(f"Per Book -> Mean: {ratings_per_book.mean():.2f}, Median: {ratings_per_book.median()}, 90th Percentile: {np.percentile(ratings_per_book, 90):.1f}, Max: {ratings_per_book.max()}")

# 5. Matrix sparsity
n_users = bronze_users["User-ID"].nunique()
n_books = bronze_books["ISBN"].nunique()
possible_pairs = n_users * n_books
actual_explicit_pairs = len(explicit_ratings)
density_pct = (actual_explicit_pairs / possible_pairs) * 100
sparsity_pct = 100.0 - density_pct
print(f"\n=== Matrix Sparsity ===")
print(f"Possible User-Book Pairs: {possible_pairs:,}")
print(f"Observed Explicit Pairs : {actual_explicit_pairs:,}")
print(f"Matrix Sparsity         : {sparsity_pct:.6f}% (Density: {density_pct:.6f}%)")


=== Users Table | Shape: (278858, 4) ===
                    Data_Type  Missing_Count  Missing_Pct
User-ID                 int64              0         0.00
Location               object              0         0.00
Age                   float64         110762        39.72
ingestion_timestamp    object              0         0.00

=== Books Table | Shape: (271360, 9) ===
                    Data_Type  Missing_Count  Missing_Pct
ISBN                   object              0          0.0
Book-Title             object              0          0.0
Book-Author            object              2          0.0
Year-Of-Publication    object              0          0.0
Publisher              object              2          0.0
Image-URL-S            object              0          0.0
Image-URL-M            object              0          0.0
Image-URL-L            object              3          0.0
ingestion_timestamp    object              0          0.0

=== Ratings Table | Shape: (1149780, 4) ===
 

 ### Bronze EDA Findings & Silver Layer Plan

* **Shape, data types, and missing values:** `Users` (278,858 rows) is missing 39.7% of `Age` values, `Books` (271,360 rows) stores `Year-Of-Publication` as mixed text/numbers with minor nulls, and `Ratings` (1,149,780 rows) has no nulls; in Silver, we will enforce proper numeric types, impute missing ages, and drop unnecessary image URL columns.
* **Distribution of ratings:** 716,109 out of 1,149,780 ratings (62.28%) are `0` (implicit interactions rather than explicit scores); in Silver, we will remove all `0` ratings and retain only explicit ratings from `1` to `10`.
* **Distribution of user ages and publication years:** User ages contain invalid outliers ranging from `0` to `244`, and book publication years contain `0`s, text strings, and future years up to `2050`; in Silver, we will set ages `< 5` or `> 100` to null and fill nulls with the median age, and set publication years `<= 0` or `> 2026` to null.
* **Number of ratings per user and per book:** Activity is heavily long-tailed with a median of `1` explicit rating per user and `1` per book (and 90th percentiles below `10`); in Silver, we will require at least `15` explicit ratings per user and per book to eliminate cold-start noise.
* **Matrix sparsity:** The user-book rating matrix is `99.9994%` sparse (only `0.00057%` of possible pairs have explicit ratings); in Silver, we will enforce referential integrity and apply the minimum rating count filter before building a sparse matrix in Gold.

In [12]:
silver_users = pd.read_parquet("bronze/users.parquet").drop(columns=["ingestion_timestamp"])
silver_books = pd.read_parquet("bronze/books.parquet").drop(columns=["ingestion_timestamp"])
silver_ratings = pd.read_parquet("bronze/ratings.parquet").drop(columns=["ingestion_timestamp"])

# 1. Clean ISBNs and text BEFORE deduplicating so case-insensitive duplicates are caught
silver_books["ISBN"] = silver_books["ISBN"].astype(str).str.strip().str.upper()
silver_ratings["ISBN"] = silver_ratings["ISBN"].astype(str).str.strip().str.upper()

# Remove duplicate rows
silver_users = silver_users.drop_duplicates(subset=["User-ID"]).copy()
silver_books = silver_books.drop_duplicates(subset=["ISBN"]).copy()
silver_ratings = silver_ratings.drop_duplicates(subset=["User-ID", "ISBN"]).copy()

# 2. Remove ratings of 0 and keep ratings from 1 to 10
silver_ratings["Book-Rating"] = pd.to_numeric(silver_ratings["Book-Rating"], errors="coerce")
silver_ratings = silver_ratings[silver_ratings["Book-Rating"].between(1, 10)].copy()

# 3. Fix user ages: set < 5 or > 100 to null, then fill nulls with median age
silver_users["Age"] = pd.to_numeric(silver_users["Age"], errors="coerce")
silver_users.loc[(silver_users["Age"] < 5) | (silver_users["Age"] > 100), "Age"] = np.nan
median_age = silver_users["Age"].median()
silver_users["Age"] = silver_users["Age"].fillna(median_age).astype(int)

# 4. Split Location into city, state, and country columns
loc_split = silver_users["Location"].fillna("").str.split(",", n=2, expand=True)
silver_users["city"] = loc_split[0].str.strip().str.title().replace("", "Unknown")
silver_users["state"] = loc_split[1].fillna("").str.strip().str.title().replace("", "Unknown")
silver_users["country"] = loc_split[2].fillna("").str.strip().str.title().replace("", "Unknown")
silver_users = silver_users.drop(columns=["Location"])

# 5. Fix publication years: set 0 or future years (> 2026) to null
silver_books["Year-Of-Publication"] = pd.to_numeric(silver_books["Year-Of-Publication"], errors="coerce")
silver_books.loc[
    (silver_books["Year-Of-Publication"] <= 0) | (silver_books["Year-Of-Publication"] > 2026),
    "Year-Of-Publication"
] = np.nan

# 6. Clean text columns & drop image-URL columns
image_cols = [col for col in silver_books.columns if "Image-URL" in col]
silver_books = silver_books.drop(columns=image_cols)

for col in ["Book-Title", "Book-Author", "Publisher"]:
    silver_books[col] = (
        silver_books[col]
        .fillna("Unknown")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.title()
    )

# Standardize column names
silver_users = silver_users.rename(columns={"User-ID": "user_id", "Age": "age"})
silver_books = silver_books.rename(columns={
    "ISBN": "isbn",
    "Book-Title": "book_title",
    "Book-Author": "book_author",
    "Year-Of-Publication": "publication_year",
    "Publisher": "publisher"
})
silver_ratings = silver_ratings.rename(columns={
    "User-ID": "user_id",
    "ISBN": "isbn",
    "Book-Rating": "rating"
})

# 7. Keep only ratings whose book and user exist in books and users tables
silver_ratings = silver_ratings[
    silver_ratings["user_id"].isin(silver_users["user_id"]) &
    silver_ratings["isbn"].isin(silver_books["isbn"])
].copy()

# 8. Keep only users and books with at least MIN_RATINGS = 10 explicit ratings
MIN_RATINGS = 10

active_users = silver_ratings["user_id"].value_counts()
active_books = silver_ratings["isbn"].value_counts()

silver_ratings = silver_ratings[
    silver_ratings["user_id"].isin(active_users[active_users >= MIN_RATINGS].index) &
    silver_ratings["isbn"].isin(active_books[active_books >= MIN_RATINGS].index)
].copy()

# Filter dimension tables to match active ratings
silver_users = silver_users[silver_users["user_id"].isin(silver_ratings["user_id"])].reset_index(drop=True)
silver_books = silver_books[silver_books["isbn"].isin(silver_ratings["isbn"])].reset_index(drop=True)
silver_ratings = silver_ratings.reset_index(drop=True)

# Save Silver tables
silver_users.to_parquet("silver/users.parquet", index=False)
silver_books.to_parquet("silver/books.parquet", index=False)
silver_ratings.to_parquet("silver/ratings.parquet", index=False)

print(f"Silver Layer Saved -> Users: {len(silver_users):,}, Books: {len(silver_books):,}, Ratings: {len(silver_ratings):,}")

Silver Layer Saved -> Users: 6,319, Books: 5,440, Ratings: 87,233


In [13]:
from scipy.sparse import csr_matrix, save_npz

# Load Silver tables
s_users = pd.read_parquet("silver/users.parquet")
s_books = pd.read_parquet("silver/books.parquet")
s_ratings = pd.read_parquet("silver/ratings.parquet")

# Build sorted contiguous integer indices for users and books
sorted_user_ids = np.sort(s_ratings["user_id"].unique())
sorted_isbns = np.sort(s_ratings["isbn"].unique())

user_to_row = {int(uid): idx for idx, uid in enumerate(sorted_user_ids)}
isbn_to_col = {str(isbn): idx for idx, isbn in enumerate(sorted_isbns)}

# 1. User-Book Sparse Rating Matrix (rows = users, cols = books)
rows = s_ratings["user_id"].map(user_to_row).values
cols = s_ratings["isbn"].map(isbn_to_col).values
vals = s_ratings["rating"].astype(np.float32).values

user_book_matrix = csr_matrix((vals, (rows, cols)), shape=(len(sorted_user_ids), len(sorted_isbns)))
save_npz("gold/user_book_matrix.npz", user_book_matrix)

# 2. Lookup Tables (book_lookup.parquet and user_lookup.parquet)
book_lookup = s_books[["isbn", "book_title", "book_author", "publication_year"]].copy()
book_lookup["book_idx"] = book_lookup["isbn"].map(isbn_to_col)
book_lookup = book_lookup.sort_values("book_idx").reset_index(drop=True)
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

user_lookup = pd.DataFrame({
    "user_id": sorted_user_ids,
    "user_idx": np.arange(len(sorted_user_ids))
}).merge(s_users, on="user_id", how="left")
user_lookup.to_parquet("gold/user_lookup.parquet", index=False)

# 3. User-Level and Book-Level Aggregates
user_aggregates = (
    s_ratings.groupby("user_id", as_index=False)
    .agg(rating_count=("rating", "count"), avg_rating=("rating", "mean"))
)
user_aggregates["avg_rating"] = user_aggregates["avg_rating"].round(3)
user_aggregates.to_parquet("gold/user_aggregates.parquet", index=False)

book_aggregates = (
    s_ratings.groupby("isbn", as_index=False)
    .agg(rating_count=("rating", "count"), avg_rating=("rating", "mean"))
    .merge(book_lookup[["isbn", "book_title", "book_author"]], on="isbn", how="left")
)
book_aggregates["avg_rating"] = book_aggregates["avg_rating"].round(3)
book_aggregates.to_parquet("gold/book_aggregates.parquet", index=False)

print(f"Gold Layer Saved -> Matrix Shape: {user_book_matrix.shape}, Non-Zero Ratings: {user_book_matrix.nnz:,}")

Gold Layer Saved -> Matrix Shape: (6319, 5440), Non-Zero Ratings: 87,233


In [14]:
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

# 1. Load Gold rating matrix and transpose so rows = books and columns = users
user_book_mat = load_npz("gold/user_book_matrix.npz").tocsr()
book_user_mat = user_book_mat.T.tocsr()

book_lookup_df = pd.read_parquet("gold/book_lookup.parquet")
user_lookup_df = pd.read_parquet("gold/user_lookup.parquet")

idx_to_book = book_lookup_df.set_index("book_idx").to_dict(orient="index")
user_id_to_idx = dict(zip(user_lookup_df["user_id"], user_lookup_df["user_idx"]))

# 2. Compute cosine similarity between books using NearestNeighbors
knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(book_user_mat)

# 3. Function returning the 10 most similar books for a given book title
def get_similar_books(book_title: str, n: int = 10) -> pd.DataFrame:
    matches = book_lookup_df[
        book_lookup_df["book_title"].str.contains(book_title, case=False, na=False, regex=False)
    ]
    if matches.empty:
        raise ValueError(f"Book title '{book_title}' not found in Gold catalog.")

    anchor = matches.iloc[0]
    anchor_idx = int(anchor["book_idx"])
    print(f"Reference Book: '{anchor['book_title']}' by {anchor['book_author']}")

    distances, indices = knn.kneighbors(book_user_mat[anchor_idx], n_neighbors=n + 1)
    similar_list = []
    for dist, b_idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
        info = idx_to_book[int(b_idx)]
        similar_list.append({
            "book_title": info["book_title"],
            "book_author": info["book_author"],
            "isbn": info["isbn"],
            "similarity": round(1.0 - float(dist), 4)
        })
    return pd.DataFrame(similar_list)

# 4. Function returning top n personalized recommendations for user_id
def recommend(user_id: int, n: int = 10) -> pd.DataFrame:
    if user_id not in user_id_to_idx:
        raise ValueError(f"User ID {user_id} not found in Gold dataset.")

    u_idx = user_id_to_idx[user_id]
    user_vector = user_book_mat[u_idx]
    already_read = set(user_vector.indices)
    rated_books = dict(zip(user_vector.indices, user_vector.data))

    # Select books the user rated highly (>= 7 or >= user's average rating)
    min_fav = max(7.0, float(np.mean(list(rated_books.values()))))
    high_rated = {b_idx: r for b_idx, r in rated_books.items() if r >= min_fav}
    if not high_rated:
        high_rated = rated_books

    weighted_scores = {}
    for b_idx, rating in high_rated.items():
        distances, neighbors = knn.kneighbors(book_user_mat[b_idx], n_neighbors=25)
        for dist, cand_idx in zip(distances.flatten()[1:], neighbors.flatten()[1:]):
            cand_idx = int(cand_idx)
            if cand_idx in already_read:
                continue
            sim = max(0.0, 1.0 - float(dist))
            weighted_scores[cand_idx] = weighted_scores.get(cand_idx, 0.0) + (sim * rating)

    top_candidates = sorted(weighted_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    output = []
    for rank, (cand_idx, score) in enumerate(top_candidates, start=1):
        info = idx_to_book[cand_idx]
        output.append({
            "rank": rank,
            "book_title": info["book_title"],
            "book_author": info["book_author"],
            "isbn": info["isbn"],
            "weighted_score": round(score, 4)
        })
    return pd.DataFrame(output)

# Quick test of Step 5.3
display(get_similar_books("Harry Potter", n=10))

Reference Book: 'Harry Potter And The Chamber Of Secrets (Book 2)' by J. K. Rowling


,book_title,book_author,isbn,similarity
0,Harry Potter And The Sorcerer'S Stone (Book 1),J. K. Rowling,0590353403,0.6800
1,Harry Potter And The Prisoner Of Azkaban (Book 3),J. K. Rowling,0439136350,0.6768
2,Harry Potter And The Goblet Of Fire (Book 4),J. K. Rowling,0439139597,0.6177
3,Harry Potter And The Order Of The Phoenix (Boo...,J. K. Rowling,043935806X,0.4008
4,Harry Potter And The Goblet Of Fire (Book 4),J. K. Rowling,0439139600,0.2059
5,Harry Potter And The Sorcerer'S Stone (Harry P...,J. K. Rowling,059035342X,0.1760
6,The Hobbit: Or There And Back Again,J.R.R. Tolkien,0618002219,0.1717
7,The Fellowship Of The Ring (The Lord Of The Ri...,J. R. R. Tolkien,0618002227,0.1580
8,The Bonesetter'S Daughter,Amy Tan,0399146431,0.1338
9,Are You My Mother?,Philip D. Eastman,0394800184,0.1320


In [16]:
import shutil
from google.colab import files

FOLDER_NAME = "LCI2023016_DEVANSH_VERMA"

for layer in ["bronze", "silver", "gold"]:
    os.makedirs(os.path.join(FOLDER_NAME, layer), exist_ok=True)
    for fname in os.listdir(layer):
        shutil.copy(os.path.join(layer, fname), os.path.join(FOLDER_NAME, layer, fname))

zip_path = shutil.make_archive(FOLDER_NAME, "zip", ".", FOLDER_NAME)
print(f"Packaged {zip_path} successfully!")
files.download(zip_path)

Packaged /content/LCI2023016_DEVANSH_VERMA.zip successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>